In [ ]:
from pathlib import Path

GRAPH_DIR = Path("./Data_ml/graph_dataset")

for f in sorted(GRAPH_DIR.glob("*hgt*")):
    print(f.name)

In [ ]:
from pathlib import Path

for p in Path(".").rglob("hgt_fold0_best.pt"):
    print(p)

In [ ]:
from pathlib import Path

MODEL_DIR = Path(
    "./Data_ml/graph_dataset/day13_explainability/saved_models"
)

for f in sorted(MODEL_DIR.glob("*.pt")):
    print(f.name)

In [ ]:
import torch

ckpt = torch.load(
    MODEL_DIR / "hgt_fold0_best.pt",
    map_location="cpu"
)

print(type(ckpt))

In [ ]:
print(ckpt.keys())

In [ ]:
print("in_dim =", ckpt["in_dim"])
print("hidden_dim =", ckpt["hidden_dim"])
print("emb_dim =", ckpt["emb_dim"])
print("heads =", ckpt["heads"])
print("dropout =", ckpt["dropout"])

print("\nmetadata:")
print(ckpt["metadata"])

In [ ]:
state = ckpt["model_state_dict"]

print(len(state))

for k in list(state.keys())[:50]:
    print(k)

In [ ]:
print(ckpt["metadata"])

In [ ]:
for k in list(state.keys())[:50]:
    print(k)

سلول ۱۳-۵ — بازسازی HGT و لود checkpoint

In [ ]:
import torch

import torch.nn as nn

import torch.nn.functional as F

import pandas as pd

import numpy as np

from pathlib import Path

from sklearn.metrics import (

    roc_auc_score,

    average_precision_score,

    f1_score,

    accuracy_score,

    precision_score,

    recall_score,

    brier_score_loss,

)

from torch_geometric.nn import HGTConv

PROJECT_ROOT = Path(".")

GRAPH_DIR = PROJECT_ROOT / "Data_ml" / "graph_dataset"

EXPLAIN_DIR = GRAPH_DIR / "day13_explainability"

MODEL_DIR = EXPLAIN_DIR / "saved_models"

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print("DEVICE:", DEVICE)

MODEL_PATH = MODEL_DIR / "hgt_fold0_best.pt"

ckpt = torch.load(MODEL_PATH, map_location="cpu")

HGT_METADATA = ckpt["metadata"]

print("checkpoint keys:", ckpt.keys())

print("metadata:", HGT_METADATA)

print("best_epoch:", ckpt["best_epoch"])

سلول ۱۳-۶ — تعریف دوباره مدل HGT دقیقاً مطابق checkpoint

In [ ]:
class HGTEncoder(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim=128,
        out_dim=128,
        heads=4,
        dropout=0.35,
    ):
        super().__init__()

        self.lin_in = nn.Linear(in_dim, hidden_dim)

        self.hgt1 = HGTConv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            metadata=HGT_METADATA,
            heads=heads,
        )

        self.hgt2 = HGTConv(
            in_channels=hidden_dim,
            out_channels=out_dim,
            metadata=HGT_METADATA,
            heads=heads,
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(out_dim)
        self.dropout = dropout

    def forward(self, x_dict, edge_index_dict):
        x = x_dict["protein"]

        x = self.lin_in(x)
        x = self.norm1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = {"protein": x}

        x_dict = self.hgt1(x_dict, edge_index_dict)
        x = x_dict["protein"]
        x = self.norm1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = {"protein": x}

        x_dict = self.hgt2(x_dict, edge_index_dict)
        x = x_dict["protein"]
        x = self.norm2(x)
        x = F.gelu(x)

        return {"protein": x}


class HeteroLinkPredictor(nn.Module):
    def __init__(self, emb_dim=128, hidden_dim=128, dropout=0.35):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 4, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, z_dict, edge_label_index):
        z = z_dict["protein"]

        src = edge_label_index[0]
        dst = edge_label_index[1]

        z_src = z[src]
        z_dst = z[dst]

        h = torch.cat(
            [
                z_src,
                z_dst,
                torch.abs(z_src - z_dst),
                z_src * z_dst,
            ],
            dim=1,
        )

        return self.mlp(h).squeeze(-1)


class HGTLinkModel(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim=128,
        emb_dim=128,
        heads=4,
        dropout=0.35,
    ):
        super().__init__()

        self.encoder = HGTEncoder(
            in_dim=in_dim,
            hidden_dim=hidden_dim,
            out_dim=emb_dim,
            heads=heads,
            dropout=dropout,
        )

        self.predictor = HeteroLinkPredictor(
            emb_dim=emb_dim,
            hidden_dim=128,
            dropout=dropout,
        )

    def forward(self, data, edge_label_index, edge_types_to_keep=None):
        if edge_types_to_keep is None:
            edge_index_dict = {
                k: v
                for k, v in data.edge_index_dict.items()
                if k in HGT_METADATA[1]
            }
        else:
            edge_index_dict = {
                k: v
                for k, v in data.edge_index_dict.items()
                if k in edge_types_to_keep
            }

        z_dict = self.encoder(
            data.x_dict,
            edge_index_dict,
        )

        logits = self.predictor(
            z_dict,
            edge_label_index,
        )

        return logits

سلول ۱۳-۷ — لود داده‌های گراف

In [ ]:
from torch_geometric.data import HeteroData

nodes = pd.read_csv(GRAPH_DIR / "graph_nodes.csv")
interaction_edges = pd.read_csv(GRAPH_DIR / "interaction_edges_labeled.csv")
ppi_edges = pd.read_csv(GRAPH_DIR / "ppi_edges.csv")
loc_edges = pd.read_csv(GRAPH_DIR / "colocalization_edges.csv")
node_features = np.load(GRAPH_DIR / "node_features_esm650.npy")

hetero_data = HeteroData()
hetero_data["protein"].x = torch.tensor(node_features, dtype=torch.float32)

def edge_index_from_df(df):
    return torch.tensor(df[["src", "dst"]].values.T, dtype=torch.long)

# enzyme-substrate
hetero_data["protein", "enzyme_substrate", "protein"].edge_index = edge_index_from_df(
    interaction_edges
)

# ppi undirected
ppi_edge_index = edge_index_from_df(ppi_edges)
ppi_edge_index = torch.cat([ppi_edge_index, ppi_edge_index[[1, 0], :]], dim=1)
hetero_data["protein", "ppi", "protein"].edge_index = ppi_edge_index

# co-localization undirected
loc_edge_index = edge_index_from_df(loc_edges)
loc_edge_index = torch.cat([loc_edge_index, loc_edge_index[[1, 0], :]], dim=1)
hetero_data["protein", "co_localized", "protein"].edge_index = loc_edge_index

# prediction labels
hetero_data["protein", "predicts", "protein"].edge_label_index = torch.tensor(
    interaction_edges[["src", "dst"]].values.T,
    dtype=torch.long,
)

hetero_data["protein", "predicts", "protein"].edge_label = torch.tensor(
    interaction_edges["edge_label"].astype(int).values,
    dtype=torch.float32,
)

hetero_data = hetero_data.to(DEVICE)

print(hetero_data)

سلول ۱۳-۸ — ساخت foldها و انتخاب fold صفر

In [ ]:
from sklearn.model_selection import GroupKFold

y = interaction_edges["edge_label"].astype(int).values
groups = interaction_edges["group_id"].astype(str).values

gkf = GroupKFold(n_splits=5)
fold_splits = []

for fold, (tr, te) in enumerate(gkf.split(interaction_edges, y, groups)):
    fold_splits.append((tr, te))
    print(fold, len(tr), len(te), y[te].sum(), len(te) - y[te].sum())

fold = 0
train_idx, test_idx = fold_splits[fold]

test_edge_index = hetero_data["protein", "predicts", "protein"].edge_label_index[:, test_idx].to(DEVICE)
test_y = hetero_data["protein", "predicts", "protein"].edge_label[test_idx].detach().cpu().numpy()

print("fold:", fold)
print("test edges:", test_edge_index.shape)

سلول ۱۳-۹ — لود مدل HGT fold0

In [ ]:
model = HGTLinkModel(
    in_dim=ckpt["in_dim"],
    hidden_dim=ckpt["hidden_dim"],
    emb_dim=ckpt["emb_dim"],
    heads=ckpt["heads"],
    dropout=ckpt["dropout"],
).to(DEVICE)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print("model loaded")

سلول ۱۳-۱۰ — تابع ارزیابی سناریوهای حذف رابطه

In [ ]:
def compute_metrics(y_true, prob):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob).astype(float)
    pred = (prob >= 0.5).astype(int)

    return {
        "roc_auc": roc_auc_score(y_true, prob),
        "pr_auc": average_precision_score(y_true, prob),
        "f1": f1_score(y_true, pred, zero_division=0),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "brier": brier_score_loss(y_true, prob),
    }


def evaluate_relation_scenario(model, data, test_edge_index, test_y, edge_types_to_keep, scenario_name):
    model.eval()

    with torch.no_grad():
        logits = model(
            data,
            test_edge_index,
            edge_types_to_keep=edge_types_to_keep,
        )

        prob = torch.sigmoid(logits).detach().cpu().numpy()

    metrics = compute_metrics(test_y, prob)
    metrics["scenario"] = scenario_name
    metrics["n_edge_types"] = len(edge_types_to_keep)

    return metrics, prob

سلول ۱۳-۱۱ — اجرای Relation Ablation روی fold0

In [ ]:
ETYPE_ES = ("protein", "enzyme_substrate", "protein")
ETYPE_PPI = ("protein", "ppi", "protein")
ETYPE_LOC = ("protein", "co_localized", "protein")

relation_scenarios = {
    "full": [ETYPE_ES, ETYPE_PPI, ETYPE_LOC],
    "no_ppi": [ETYPE_ES, ETYPE_LOC],
    "no_colocalization": [ETYPE_ES, ETYPE_PPI],
    "no_enzyme_substrate_context": [ETYPE_PPI, ETYPE_LOC],
    "only_enzyme_substrate": [ETYPE_ES],
    "only_ppi": [ETYPE_PPI],
    "only_colocalization": [ETYPE_LOC],
}

relation_rows = []
relation_probs = {}

for scenario_name, edge_types in relation_scenarios.items():
    print("Running:", scenario_name)

    metrics, prob = evaluate_relation_scenario(
        model=model,
        data=hetero_data,
        test_edge_index=test_edge_index,
        test_y=test_y,
        edge_types_to_keep=edge_types,
        scenario_name=scenario_name,
    )

    relation_rows.append(metrics)
    relation_probs[scenario_name] = prob

relation_ablation_fold0 = pd.DataFrame(relation_rows)

full_pr = relation_ablation_fold0.loc[
    relation_ablation_fold0["scenario"] == "full",
    "pr_auc"
].iloc[0]

full_roc = relation_ablation_fold0.loc[
    relation_ablation_fold0["scenario"] == "full",
    "roc_auc"
].iloc[0]

relation_ablation_fold0["delta_pr_vs_full"] = relation_ablation_fold0["pr_auc"] - full_pr
relation_ablation_fold0["delta_roc_vs_full"] = relation_ablation_fold0["roc_auc"] - full_roc

relation_ablation_fold0 = relation_ablation_fold0.sort_values(
    "pr_auc",
    ascending=False,
)

display(relation_ablation_fold0)

relation_ablation_fold0.to_csv(
    EXPLAIN_DIR / "relation_ablation_fold0.csv",
    index=False,
)

سلول ۱۳-۱۲

ساخت گراف همسایگی

In [ ]:
from collections import defaultdict

neighbor_graph = defaultdict(set)

for df in [interaction_edges, ppi_edges, loc_edges]:

    for _, r in df.iterrows():

        s = int(r["src"])
        d = int(r["dst"])

        neighbor_graph[s].add(d)
        neighbor_graph[d].add(s)

print("nodes with neighbors:", len(neighbor_graph))

سلول ۱۳-۱۳

نگاشت node_id → gene

In [ ]:
node2gene = (
    nodes
    .set_index("node_id")["gene"]
    .fillna("UNKNOWN")
    .to_dict()
)

node2ac = (
    nodes
    .set_index("node_id")["uniprot_ac"]
    .to_dict()
)



سلول ۱۳-۱۴

پیدا کردن قوی‌ترین Positive Predictionها

In [ ]:
pred_df = pd.read_csv(
    EXPLAIN_DIR / "hgt_saved_fold_predictions.csv"
)

pred_df = pred_df.sort_values(
    "prob_hgt",
    ascending=False
)

display(
    pred_df[
        [
            "enz_gene",
            "sub_gene",
            "prob_hgt",
            "y_true"
        ]
    ].head(20)
)

In [ ]:
pred_df = pd.read_csv(
    EXPLAIN_DIR / "hgt_saved_fold_predictions.csv"
)

print(pred_df.columns.tolist())
print(pred_df.shape)

In [ ]:
pred_df = pd.read_csv(
    EXPLAIN_DIR / "hgt_saved_fold_predictions.csv"
)

pred_df = pred_df.sort_values(
    "prob_hgt_saved",
    ascending=False
)

display(
    pred_df[
        [
            "enz_gene",
            "sub_gene",
            "prob_hgt_saved",
            "y_true"
        ]
    ].head(20)
)

In [ ]:
pred_df.head(20)

سلول ۱۳-۱۵

انتخاب یک interaction

In [ ]:
TARGET_ROW = 0

target = pred_df.iloc[TARGET_ROW]

display(target)



سلول ۱۳-۱۶

پیدا کردن nodeها

In [ ]:
enz_ac = target["enz_ac"]
sub_ac = target["sub_ac"]

enz_node = int(
    nodes.loc[
        nodes["uniprot_ac"] == enz_ac,
        "node_id"
    ].iloc[0]
)

sub_node = int(
    nodes.loc[
        nodes["uniprot_ac"] == sub_ac,
        "node_id"
    ].iloc[0]
)

print("enzyme node:", enz_node)
print("substrate node:", sub_node)



سلول ۱۳-۱۷

مهم‌ترین همسایه‌های دو طرف

In [ ]:
enz_neighbors = list(neighbor_graph[enz_node])
sub_neighbors = list(neighbor_graph[sub_node])

print("enzyme neighbors:", len(enz_neighbors))
print("substrate neighbors:", len(sub_neighbors))

سلول ۱۳-۱۸

رتبه‌بندی همسایه‌ها با degree

فعلاً ساده‌ترین و پایدارترین Importance را می‌گیریم.

In [ ]:
neighbor_scores = []

candidate_neighbors = (
    set(enz_neighbors)
    |
    set(sub_neighbors)
)

for n in candidate_neighbors:

    score = len(neighbor_graph[n])

    neighbor_scores.append(
        {
            "node_id": n,
            "gene": node2gene.get(n, "UNKNOWN"),
            "uniprot": node2ac.get(n, ""),
            "degree": score,
        }
    )

neighbor_scores = (
    pd.DataFrame(neighbor_scores)
    .sort_values(
        "degree",
        ascending=False
    )
)

display(neighbor_scores.head(20))

مرحله بعد: Node Importance واقعی با Perturbation

این کد را اجرا کن.

سلول ۱۳-۱۹ — انتخاب target و لود مدل fold همان target

In [ ]:
TARGET_ROW = 0

target = pred_df.iloc[TARGET_ROW]
target_fold = int(target["fold"])

print("Target:")
display(target)

print("target_fold:", target_fold)

MODEL_PATH = MODEL_DIR / f"hgt_fold{target_fold}_best.pt"

ckpt_target = torch.load(
    MODEL_PATH,
    map_location="cpu"
)

model_target = HGTLinkModel(
    in_dim=ckpt_target["in_dim"],
    hidden_dim=ckpt_target["hidden_dim"],
    emb_dim=ckpt_target["emb_dim"],
    heads=ckpt_target["heads"],
    dropout=ckpt_target["dropout"],
).to(DEVICE)

model_target.load_state_dict(ckpt_target["model_state_dict"])
model_target.eval()

print("loaded:", MODEL_PATH)

سلول ۱۳-۲۰ — پیدا کردن edge هدف

In [ ]:
target_pair_id = target["pair_id"]

target_idx = interaction_edges.index[
    interaction_edges["pair_id"] == target_pair_id
].tolist()[0]

target_edge_index = hetero_data["protein", "predicts", "protein"].edge_label_index[
    :, target_idx:target_idx+1
].to(DEVICE)

target_y = int(
    hetero_data["protein", "predicts", "protein"].edge_label[target_idx].item()
)

print("target_idx:", target_idx)
print("target_y:", target_y)
print("target_edge_index:", target_edge_index)

سلول ۱۳-۲۱ — احتمال پایه مدل

In [ ]:
with torch.no_grad():
    base_logit = model_target(
        hetero_data,
        target_edge_index,
        edge_types_to_keep=[ETYPE_ES, ETYPE_PPI, ETYPE_LOC],
    )
    
    base_prob = torch.sigmoid(base_logit).item()

print("base_prob:", base_prob)

سلول ۱۳-۲۲ — ساخت candidate nodeها در اطراف enzyme و substrate

In [ ]:
enz_ac = target["enz_ac"]
sub_ac = target["sub_ac"]

enz_node = int(
    nodes.loc[nodes["uniprot_ac"] == enz_ac, "node_id"].iloc[0]
)

sub_node = int(
    nodes.loc[nodes["uniprot_ac"] == sub_ac, "node_id"].iloc[0]
)

candidate_nodes = list(
    set(neighbor_graph[enz_node])
    | set(neighbor_graph[sub_node])
    | {enz_node, sub_node}
)

print("enzyme:", target["enz_gene"], enz_node)
print("substrate:", target["sub_gene"], sub_node)
print("candidate_nodes:", len(candidate_nodes))

سلول ۱۳-۲۳ — Node masking importance

In [ ]:
def score_with_masked_node(model, data, edge_label_index, node_id):
    data_masked = data.clone()
    
    x = data_masked["protein"].x.clone()
    
    # mask کامل featureهای آن node
    x[node_id, :] = 0.0
    
    data_masked["protein"].x = x
    
    with torch.no_grad():
        logit = model(
            data_masked,
            edge_label_index,
            edge_types_to_keep=[ETYPE_ES, ETYPE_PPI, ETYPE_LOC],
        )
        
        prob = torch.sigmoid(logit).item()
    
    return prob


node_importance_rows = []

for n in candidate_nodes:
    masked_prob = score_with_masked_node(
        model=model_target,
        data=hetero_data,
        edge_label_index=target_edge_index,
        node_id=n,
    )
    
    importance = base_prob - masked_prob
    
    node_importance_rows.append({
        "pair_id": target_pair_id,
        "target_enzyme": target["enz_gene"],
        "target_substrate": target["sub_gene"],
        "node_id": n,
        "gene": node2gene.get(n, "UNKNOWN"),
        "uniprot": node2ac.get(n, ""),
        "degree": len(neighbor_graph[n]),
        "base_prob": base_prob,
        "masked_prob": masked_prob,
        "importance_drop": importance,
    })

node_importance = (
    pd.DataFrame(node_importance_rows)
    .sort_values("importance_drop", ascending=False)
)

display(node_importance.head(30))

node_importance.to_csv(
    EXPLAIN_DIR / f"node_importance_{target_pair_id.replace('|','_')}.csv",
    index=False
)

سلول ۱۳-۲۴

انتخاب Top 50 Positive Prediction

In [ ]:
TOP_K = 50

top_pairs = (
    pred_df
    .sort_values(
        "prob_hgt_saved",
        ascending=False
    )
    .head(TOP_K)
    .reset_index(drop=True)
)

display(
    top_pairs[
        [
            "enz_gene",
            "sub_gene",
            "prob_hgt_saved"
        ]
    ].head()
)

print("pairs:", len(top_pairs))

سلول ۱۳-۲۵

تابع Importance برای یک زوج

In [ ]:
def compute_pair_node_importance(
    target_row,
    top_n_neighbors=50
):
    
    target_pair_id = target_row["pair_id"]

    target_idx = interaction_edges.index[
        interaction_edges["pair_id"] == target_pair_id
    ].tolist()[0]

    edge_idx = hetero_data[
        "protein",
        "predicts",
        "protein"
    ].edge_label_index[
        :,
        target_idx:target_idx+1
    ].to(DEVICE)

    fold = int(target_row["fold"])

    ckpt = torch.load(
        MODEL_DIR / f"hgt_fold{fold}_best.pt",
        map_location="cpu"
    )

    model = HGTLinkModel(
        in_dim=ckpt["in_dim"],
        hidden_dim=ckpt["hidden_dim"],
        emb_dim=ckpt["emb_dim"],
        heads=ckpt["heads"],
        dropout=ckpt["dropout"]
    ).to(DEVICE)

    model.load_state_dict(
        ckpt["model_state_dict"]
    )

    model.eval()

    with torch.no_grad():

        base_prob = torch.sigmoid(
            model(
                hetero_data,
                edge_idx,
                edge_types_to_keep=[
                    ETYPE_ES,
                    ETYPE_PPI,
                    ETYPE_LOC
                ]
            )
        ).item()

    enz_node = int(
        nodes.loc[
            nodes["uniprot_ac"]
            == target_row["enz_ac"],
            "node_id"
        ].iloc[0]
    )

    sub_node = int(
        nodes.loc[
            nodes["uniprot_ac"]
            == target_row["sub_ac"],
            "node_id"
        ].iloc[0]
    )

    candidate_nodes = list(
        set(neighbor_graph[enz_node])
        |
        set(neighbor_graph[sub_node])
        |
        {enz_node, sub_node}
    )

    importance_rows = []

    for n in candidate_nodes:

        x_backup = (
            hetero_data["protein"]
            .x[n]
            .clone()
        )

        hetero_data["protein"].x[n] = 0

        with torch.no_grad():

            masked_prob = torch.sigmoid(
                model(
                    hetero_data,
                    edge_idx,
                    edge_types_to_keep=[
                        ETYPE_ES,
                        ETYPE_PPI,
                        ETYPE_LOC
                    ]
                )
            ).item()

        hetero_data["protein"].x[n] = x_backup

        importance_rows.append({
            "gene": node2gene.get(n, "UNK"),
            "importance":
                base_prob - masked_prob
        })

    imp_df = (
        pd.DataFrame(importance_rows)
        .sort_values(
            "importance",
            ascending=False
        )
        .head(top_n_neighbors)
    )

    return imp_df

سلول ۱۳-۲۶

محاسبه Global Importance

In [ ]:
all_importance = []

for i in range(len(top_pairs)):

    print(
        f"{i+1}/{len(top_pairs)}"
    )

    imp_df = compute_pair_node_importance(
        top_pairs.iloc[i]
    )

    all_importance.append(
        imp_df
    )

global_importance = pd.concat(
    all_importance,
    ignore_index=True
)

print(global_importance.shape)



سلول ۱۳-۲۷

تجمیع نهایی

In [ ]:
global_node_importance = (
    global_importance
    .groupby("gene")
    .agg(
        frequency=(
            "importance",
            "count"
        ),
        mean_importance=(
            "importance",
            "mean"
        ),
        total_importance=(
            "importance",
            "sum"
        )
    )
    .reset_index()
    .sort_values(
        "total_importance",
        ascending=False
    )
)

display(
    global_node_importance
    .head(50)
)

global_node_importance.to_csv(
    EXPLAIN_DIR /
    "global_node_importance.csv",
    index=False
)

سلول ۱۳-۲۸

In [ ]:
import matplotlib.pyplot as plt

top20 = (
    global_node_importance
    .head(20)
    .iloc[::-1]
)

plt.figure(
    figsize=(8,7)
)

plt.barh(
    top20["gene"],
    top20["total_importance"]
)

plt.xlabel(
    "Total Importance"
)

plt.ylabel(
    "Gene"
)

plt.title(
    "Global Node Importance (HGT)"
)

plt.tight_layout()

plt.show()

سلول ۱۳-۲۹ — لود Relation Ablation

In [ ]:
relation_ablation = pd.read_csv(
    EXPLAIN_DIR / "relation_ablation_fold0.csv"
)

display(relation_ablation)

سلول ۱۳-۳۰ — محاسبه Contribution نسبی هر Relation

In [ ]:
full_pr = relation_ablation.loc[
    relation_ablation["scenario"] == "full",
    "pr_auc"
].iloc[0]

drops = {
    "PPI": full_pr - relation_ablation.loc[
        relation_ablation["scenario"] == "no_ppi",
        "pr_auc"
    ].iloc[0],

    "Co-localization": full_pr - relation_ablation.loc[
        relation_ablation["scenario"] == "no_colocalization",
        "pr_auc"
    ].iloc[0],

    "Enzyme-Substrate Context": full_pr - relation_ablation.loc[
        relation_ablation["scenario"] == "no_enzyme_substrate_context",
        "pr_auc"
    ].iloc[0],
}

edge_importance = pd.DataFrame([
    {
        "relation": k,
        "pr_auc_drop": v
    }
    for k, v in drops.items()
])

edge_importance["relative_importance"] = (
    edge_importance["pr_auc_drop"]
    / edge_importance["pr_auc_drop"].sum()
)

edge_importance["relative_importance_percent"] = (
    edge_importance["relative_importance"] * 100
)

edge_importance = edge_importance.sort_values(
    "relative_importance_percent",
    ascending=False
)

display(edge_importance)

edge_importance.to_csv(
    EXPLAIN_DIR / "edge_type_importance_from_ablation.csv",
    index=False
)

سلول ۱۳-۳۱ — Figure مقاله‌ای Edge-Type Importance

In [ ]:
import matplotlib.pyplot as plt

plot_df = edge_importance.sort_values(
    "relative_importance_percent",
    ascending=True
)

plt.figure(figsize=(8, 4.8))

plt.barh(
    plot_df["relation"],
    plot_df["relative_importance_percent"]
)

plt.xlabel("Relative Importance (%)")
plt.ylabel("Relation Type")
plt.title("Edge-Type Importance in HGT")

for i, v in enumerate(plot_df["relative_importance_percent"]):
    plt.text(
        v + 1,
        i,
        f"{v:.1f}%",
        va="center"
    )

plt.tight_layout()

plt.savefig(
    EXPLAIN_DIR / "figure_edge_type_importance_hgt.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    EXPLAIN_DIR / "figure_edge_type_importance_hgt.pdf",
    bbox_inches="tight"
)

plt.show()

سلول ۱۳-۳۲ — Figure افت PR-AUC در حالت حذف هر رابطه

In [ ]:
drop_df = relation_ablation[
    relation_ablation["scenario"].isin([
        "no_ppi",
        "no_colocalization",
        "no_enzyme_substrate_context"
    ])
].copy()

drop_df["removed_relation"] = drop_df["scenario"].map({
    "no_ppi": "PPI",
    "no_colocalization": "Co-localization",
    "no_enzyme_substrate_context": "Enzyme-Substrate Context"
})

drop_df["pr_auc_drop"] = full_pr - drop_df["pr_auc"]

drop_df = drop_df.sort_values("pr_auc_drop", ascending=True)

plt.figure(figsize=(8, 4.8))

plt.barh(
    drop_df["removed_relation"],
    drop_df["pr_auc_drop"]
)

plt.xlabel("PR-AUC Drop After Removal")
plt.ylabel("Removed Relation")
plt.title("Performance Drop by Relation Removal")

for i, v in enumerate(drop_df["pr_auc_drop"]):
    plt.text(
        v + 0.003,
        i,
        f"{v:.3f}",
        va="center"
    )

plt.tight_layout()

plt.savefig(
    EXPLAIN_DIR / "figure_relation_ablation_pr_drop_hgt.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    EXPLAIN_DIR / "figure_relation_ablation_pr_drop_hgt.pdf",
    bbox_inches="tight"
)

plt.show()



سلول ۱۳-۳۳

بررسی پارامترهای Relation-Specific

In [ ]:
state = ckpt["model_state_dict"]

for k in state.keys():

    if "p_rel" in k:
        print(k)

سلول ۱۳-۳۴

استخراج قدرت هر Relation

In [ ]:
relation_scores = []

for k,v in state.items():

    if "p_rel" not in k:
        continue

    rel_name = (
        k
        .split("protein__")[1]
        .split("__protein")[0]
    )

    score = (
        v.abs()
        .mean()
        .item()
    )

    relation_scores.append({
        "relation": rel_name,
        "weight_strength": score
    })

relation_scores = pd.DataFrame(
    relation_scores
)

relation_scores

سلول ۱۳-۳۵

تجمیع بین دو لایه HGT

In [ ]:
relation_attention = (
    relation_scores
    .groupby("relation")
    .agg(
        mean_strength=(
            "weight_strength",
            "mean"
        )
    )
    .reset_index()
)

relation_attention[
    "attention_percent"
] = (
    relation_attention["mean_strength"]
    /
    relation_attention["mean_strength"].sum()
    *100
)

relation_attention = (
    relation_attention
    .sort_values(
        "attention_percent",
        ascending=False
    )
)

display(
    relation_attention
)

سلول ۱۳-۳۶

In [ ]:
import matplotlib.pyplot as plt

plot_df = (
    relation_attention
    .sort_values(
        "attention_percent"
    )
)

plt.figure(
    figsize=(7,4)
)

plt.barh(
    plot_df["relation"],
    plot_df["attention_percent"]
)

for i,v in enumerate(
    plot_df["attention_percent"]
):
    plt.text(
        v+0.5,
        i,
        f"{v:.1f}%"
    )

plt.xlabel(
    "Attention Contribution (%)"
)

plt.ylabel(
    "Relation Type"
)

plt.title(
    "Learned Relation Importance in HGT"
)

plt.tight_layout()

plt.show()

سلول ۱۳-۳۷ — انتخاب Case Studyها

In [ ]:
case_candidates = [
    ("USP10", "TP53"),
    ("STUB1", "TP53"),
    ("CYLD", "TP53"),
    ("ITCH", "NOTCH1"),
    ("ITCH", "DVL2"),
    ("ITCH", "RIPK1"),
    ("VHL", "DVL2"),
    ("USP8", "ITCH"),
    ("USP8", "CASP1"),
    ("DTL", "TP53"),
]

case_rows = []

for enz_gene, sub_gene in case_candidates:
    hit = pred_df[
        (pred_df["enz_gene"].astype(str) == enz_gene)
        &
        (pred_df["sub_gene"].astype(str) == sub_gene)
    ].copy()
    
    if len(hit) == 0:
        case_rows.append({
            "enz_gene": enz_gene,
            "sub_gene": sub_gene,
            "found": False,
            "pair_id": "",
            "prob_hgt_saved": np.nan,
            "y_true": np.nan,
            "fold": np.nan,
        })
    else:
        r = hit.sort_values("prob_hgt_saved", ascending=False).iloc[0]
        case_rows.append({
            "enz_gene": enz_gene,
            "sub_gene": sub_gene,
            "found": True,
            "pair_id": r["pair_id"],
            "prob_hgt_saved": r["prob_hgt_saved"],
            "y_true": r["y_true"],
            "fold": r["fold"],
        })

case_study_table = pd.DataFrame(case_rows)
display(case_study_table)

case_study_table.to_csv(
    EXPLAIN_DIR / "case_study_candidates.csv",
    index=False
)

سلول ۱۳-۳۸ — تابع کامل Explain برای هر Case

In [ ]:
def explain_case_by_pair_id(pair_id, top_n=20):
    target_row = pred_df[pred_df["pair_id"] == pair_id].iloc[0]
    
    imp_df = compute_pair_node_importance(
        target_row,
        top_n_neighbors=top_n
    )
    
    imp_df["pair_id"] = pair_id
    imp_df["enz_gene"] = target_row["enz_gene"]
    imp_df["sub_gene"] = target_row["sub_gene"]
    imp_df["prob_hgt_saved"] = target_row["prob_hgt_saved"]
    imp_df["y_true"] = target_row["y_true"]
    
    return imp_df

سلول ۱۳-۳۹ — اجرای Explainability برای Caseهای پیدا شده

In [ ]:
all_case_explanations = []

found_cases = case_study_table[case_study_table["found"] == True].copy()

for _, r in found_cases.iterrows():
    print("Explaining:", r["enz_gene"], "->", r["sub_gene"])
    
    imp = explain_case_by_pair_id(
        r["pair_id"],
        top_n=20
    )
    
    all_case_explanations.append(imp)

case_explanations = pd.concat(
    all_case_explanations,
    ignore_index=True
)

display(case_explanations.head(50))

case_explanations.to_csv(
    EXPLAIN_DIR / "case_study_node_importance.csv",
    index=False
)

سلول ۱۳-۴۰ — خلاصه مقاله‌ای Caseها

In [ ]:
case_summary_rows = []

for pair_id, g in case_explanations.groupby("pair_id"):
    first = g.iloc[0]
    
    top_genes = (
        g.sort_values("importance", ascending=False)
        .head(5)["gene"]
        .tolist()
    )
    
    case_summary_rows.append({
        "pair_id": pair_id,
        "enzyme": first["enz_gene"],
        "substrate": first["sub_gene"],
        "probability": first["prob_hgt_saved"],
        "y_true": first["y_true"],
        "top_explanatory_genes": "; ".join(top_genes),
    })

case_summary = pd.DataFrame(case_summary_rows).sort_values(
    "probability",
    ascending=False
)

display(case_summary)

case_summary.to_csv(
    EXPLAIN_DIR / "case_study_summary.csv",
    index=False
)



سلول ۱۳-۴۱ — Figure برای یک Case مشخص

اول یکی از Caseها را انتخاب کن. مثلاً USP10 -> TP53:

In [ ]:
CASE_PAIR_ID = case_summary.iloc[0]["pair_id"]

case_plot_df = (
    case_explanations[
        case_explanations["pair_id"] == CASE_PAIR_ID
    ]
    .sort_values("importance", ascending=False)
    .head(15)
    .iloc[::-1]
)

display(case_plot_df)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.barh(
    case_plot_df["gene"],
    case_plot_df["importance"]
)

title = (
    case_plot_df["enz_gene"].iloc[0]
    + " → "
    + case_plot_df["sub_gene"].iloc[0]
)

plt.xlabel("Node Importance Drop")
plt.ylabel("Gene")
plt.title(f"Node Importance for {title}")

plt.tight_layout()

safe_title = title.replace(" → ", "_to_").replace("/", "_")

plt.savefig(
    EXPLAIN_DIR / f"figure_case_node_importance_{safe_title}.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    EXPLAIN_DIR / f"figure_case_node_importance_{safe_title}.pdf",
    bbox_inches="tight"
)

plt.show()

می‌خواهیم یک شبکه رسم کنیم که نشان دهد:

USP10 → TP53

و نودهای مهم اطرافش چه بوده‌اند.

شکل شبیه مقالات Nature Biotechnology.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

pair_id = "DUB|Q14694|P04637"

tmp = case_explanations[
    case_explanations["pair_id"] == pair_id
].copy()

top_nodes = (
    tmp.sort_values(
        "importance",
        ascending=False
    )
    .head(15)
)

G = nx.Graph()

target_enz = "USP10"
target_sub = "TP53"

G.add_node(target_enz)
G.add_node(target_sub)

G.add_edge(target_enz, target_sub)

for _, r in top_nodes.iterrows():

    gene = r["gene"]

    if gene not in [target_enz, target_sub]:

        G.add_node(gene)

        G.add_edge(gene, target_enz)

        G.add_edge(gene, target_sub)

plt.figure(figsize=(10,8))

pos = nx.spring_layout(
    G,
    seed=42,
    k=1.2
)

nx.draw_networkx_nodes(
    G,
    pos,
    node_size=1200
)

nx.draw_networkx_edges(
    G,
    pos,
    alpha=0.6
)

nx.draw_networkx_labels(
    G,
    pos,
    font_size=9
)

plt.title(
    "HGT Explanation Network: USP10 -> TP53"
)

plt.axis("off")

plt.tight_layout()

plt.savefig(
    EXPLAIN_DIR /
    "figure_case_network_USP10_TP53.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

pair_id = "DUB|Q14694|P04637"

tmp = (
    case_explanations[
        case_explanations["pair_id"] == pair_id
    ]
    .sort_values("importance", ascending=False)
    .head(15)
)

G = nx.Graph()

target_enz = "USP10"
target_sub = "TP53"

G.add_node(target_enz)
G.add_node(target_sub)

G.add_edge(target_enz, target_sub)

for _, row in tmp.iterrows():

    gene = row["gene"]

    if gene in [target_enz, target_sub]:
        continue

    G.add_node(gene)

    G.add_edge(
        target_enz,
        gene,
        weight=row["importance"]
    )

    G.add_edge(
        target_sub,
        gene,
        weight=row["importance"]
    )

# ----------------------------
# Manual radial layout
# ----------------------------

pos = {}

pos[target_enz] = (-0.3, 0)
pos[target_sub] = (0.3, 0)

neighbors = [
    n for n in G.nodes()
    if n not in [target_enz, target_sub]
]

angles = np.linspace(
    0,
    2*np.pi,
    len(neighbors),
    endpoint=False
)

radius = 2.5

for n, a in zip(neighbors, angles):

    pos[n] = (
        radius*np.cos(a),
        radius*np.sin(a)
    )

# ----------------------------
# draw
# ----------------------------

plt.figure(figsize=(14,12))

nx.draw_networkx_edges(
    G,
    pos,
    alpha=0.35,
    width=1.5
)

nx.draw_networkx_nodes(
    G,
    pos,
    nodelist=[target_enz],
    node_size=3500
)

nx.draw_networkx_nodes(
    G,
    pos,
    nodelist=[target_sub],
    node_size=3500
)

nx.draw_networkx_nodes(
    G,
    pos,
    nodelist=neighbors,
    node_size=1800
)

nx.draw_networkx_labels(
    G,
    pos,
    font_size=11,
    font_weight="bold"
)

plt.title(
    "HGT Explanation Network\nUSP10 → TP53",
    fontsize=18
)

plt.axis("off")

plt.tight_layout()

plt.savefig(
    EXPLAIN_DIR /
    "figure_case_network_USP10_TP53_v2.png",
    dpi=400,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

pair_id = "DUB|Q14694|P04637"

tmp = (
    case_explanations[
        case_explanations["pair_id"] == pair_id
    ]
    .sort_values("importance", ascending=False)
    .head(15)
    .copy()
)

target_enz = "USP10"
target_sub = "TP53"

G = nx.Graph()

G.add_node(target_enz, importance=float(tmp.loc[tmp["gene"] == target_enz, "importance"].max()))
G.add_node(target_sub, importance=float(tmp.loc[tmp["gene"] == target_sub, "importance"].max()))

G.add_edge(target_enz, target_sub, weight=1.0)

for _, row in tmp.iterrows():
    gene = row["gene"]
    imp = float(row["importance"])

    if gene in [target_enz, target_sub]:
        continue

    G.add_node(gene, importance=imp)
    G.add_edge(target_enz, gene, weight=imp)
    G.add_edge(target_sub, gene, weight=imp)

# ----------------------------
# radial layout
# ----------------------------

pos = {
    target_enz: (-0.35, 0),
    target_sub: (0.35, 0),
}

neighbors = [
    n for n in G.nodes()
    if n not in [target_enz, target_sub]
]

angles = np.linspace(0, 2 * np.pi, len(neighbors), endpoint=False)
radius = 2.7

for n, a in zip(neighbors, angles):
    pos[n] = (
        radius * np.cos(a),
        radius * np.sin(a)
    )

# ----------------------------
# node sizes from importance
# ----------------------------

importances = np.array([
    G.nodes[n].get("importance", 0.0)
    for n in G.nodes()
])

imp_min = importances.min()
imp_max = importances.max()

def scale_node_size(imp, min_size=900, max_size=4200):
    if imp_max == imp_min:
        return (min_size + max_size) / 2

    return min_size + (
        (imp - imp_min) / (imp_max - imp_min)
    ) * (max_size - min_size)

node_sizes = [
    scale_node_size(G.nodes[n].get("importance", 0.0))
    for n in G.nodes()
]

# مرکزی‌ها کمی برجسته‌تر
node_sizes = [
    max(s, 3900) if n in [target_enz, target_sub] else s
    for n, s in zip(G.nodes(), node_sizes)
]

# ----------------------------
# edge widths from importance
# ----------------------------

edge_weights = np.array([
    G.edges[e].get("weight", 0.0)
    for e in G.edges()
])

ew_min = edge_weights.min()
ew_max = edge_weights.max()

def scale_edge_width(w, min_width=0.8, max_width=4.0):
    if ew_max == ew_min:
        return 1.5

    return min_width + (
        (w - ew_min) / (ew_max - ew_min)
    ) * (max_width - min_width)

edge_widths = [
    scale_edge_width(G.edges[e].get("weight", 0.0))
    for e in G.edges()
]

# ----------------------------
# draw
# ----------------------------

plt.figure(figsize=(14, 12))

nx.draw_networkx_edges(
    G,
    pos,
    width=edge_widths,
    alpha=0.35
)

nx.draw_networkx_nodes(
    G,
    pos,
    node_size=node_sizes,
    linewidths=1.8,
    edgecolors="black"
)

nx.draw_networkx_labels(
    G,
    pos,
    font_size=10,
    font_weight="bold"
)

plt.title(
    "HGT Explanation Network: USP10 → TP53\nNode size proportional to perturbation importance",
    fontsize=18
)

plt.axis("off")
plt.tight_layout()

plt.savefig(
    EXPLAIN_DIR / "figure_case_network_USP10_TP53_importance_scaled.png",
    dpi=400,
    bbox_inches="tight"
)

plt.savefig(
    EXPLAIN_DIR / "figure_case_network_USP10_TP53_importance_scaled.pdf",
    bbox_inches="tight"
)

plt.show()

۱) پرتکرارترین E3ها در Top Predictions

In [ ]:
top_n = 500

top_preds = (
    pred_df
    .sort_values(
        "prob_hgt_saved",
        ascending=False
    )
    .head(top_n)
)

e3_counts = (
    top_preds[
        top_preds["enzyme_class"] == "E3"
    ]["enz_gene"]
    .value_counts()
    .reset_index()
)

e3_counts.columns = [
    "E3",
    "count"
]

display(e3_counts.head(30))

شکل مقاله‌ای E3

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plot_df = e3_counts.head(15)

plt.barh(
    plot_df["E3"][::-1],
    plot_df["count"][::-1]
)

plt.title(
    f"Most Frequent E3 Ligases in Top {top_n} Predictions"
)

plt.xlabel("Frequency")

plt.tight_layout()

plt.savefig(
    EXPLAIN_DIR /
    "top_E3_predictions.png",
    dpi=300
)

plt.show()

۲) پرتکرارترین DUBها

In [ ]:
dub_counts = (
    top_preds[
        top_preds["enzyme_class"] == "DUB"
    ]["enz_gene"]
    .value_counts()
    .reset_index()
)

dub_counts.columns = [
    "DUB",
    "count"
]

display(
    dub_counts.head(30)
)

شکل DUB

In [ ]:
plt.figure(figsize=(10,6))

plot_df = dub_counts.head(15)

plt.barh(
    plot_df["DUB"][::-1],
    plot_df["count"][::-1]
)

plt.title(
    f"Most Frequent DUBs in Top {top_n} Predictions"
)

plt.xlabel("Frequency")

plt.tight_layout()

plt.savefig(
    EXPLAIN_DIR /
    "top_DUB_predictions.png",
    dpi=300
)

plt.show()

۳) پرتکرارترین Substrateها

In [ ]:
substrate_counts = (
    top_preds["sub_gene"]
    .value_counts()
    .reset_index()
)

substrate_counts.columns = [
    "substrate",
    "count"
]

display(
    substrate_counts.head(30)
)

شکل Substrate

In [ ]:
plt.figure(figsize=(10,6))

plot_df = substrate_counts.head(20)

plt.barh(
    plot_df["substrate"][::-1],
    plot_df["count"][::-1]
)

plt.title(
    f"Most Frequent Substrates in Top {top_n} Predictions"
)

plt.xlabel("Frequency")

plt.tight_layout()

plt.savefig(
    EXPLAIN_DIR /
    "top_substrates_predictions.png",
    dpi=300
)

plt.show()

۴) درصد تعاملات شناخته‌شده در Top Predictions

این مهم‌ترین تحلیل Day 13 است.

In [ ]:
for n in [50,100,200,500,1000]:

    tmp = (
        pred_df
        .sort_values(
            "prob_hgt_saved",
            ascending=False
        )
        .head(n)
    )

    known_rate = (
        tmp["y_true"]
        .mean()
        * 100
    )

    print(
        f"Top {n}: "
        f"{known_rate:.2f}% known interactions"
    )

۵) جدول مقاله‌ای نهایی

In [ ]:
summary_table = pd.DataFrame({

    "metric":[
        "Top50",
        "Top100",
        "Top200",
        "Top500",
        "Top1000"
    ],

    "known_interaction_rate":[
        pred_df.sort_values(
            "prob_hgt_saved",
            ascending=False
        ).head(50)["y_true"].mean(),

        pred_df.sort_values(
            "prob_hgt_saved",
            ascending=False
        ).head(100)["y_true"].mean(),

        pred_df.sort_values(
            "prob_hgt_saved",
            ascending=False
        ).head(200)["y_true"].mean(),

        pred_df.sort_values(
            "prob_hgt_saved",
            ascending=False
        ).head(500)["y_true"].mean(),

        pred_df.sort_values(
            "prob_hgt_saved",
            ascending=False
        ).head(1000)["y_true"].mean()
    ]
})

summary_table["known_interaction_rate"] *= 100

display(summary_table)

summary_table.to_csv(
    EXPLAIN_DIR /
    "top_prediction_enrichment.csv",
    index=False
)